# Bronze Layer - Data Ingestion

In [0]:
# ============================================================
# Notebook : 01_bronze_data_ingestion
# Project  : Supply Chain Control Tower
# Author   : Nisha Sorallikar
# Purpose  : Ingest raw source data, perform data profiling,
#            standardize column names, and create the Bronze
#            Delta table.
# ============================================================

In [0]:
# =====================================================
# Notebook : 01_bronze_data_ingestion
# Project  : Supply Chain Control Tower
# Author   : Nisha Sorallikar
# Purpose  : Read and validate the source dataset.
# =====================================================

# ---------------------------------------
# Step 1 : Read Source File
# ---------------------------------------

df = (
    spark.read
         .format("csv")
         .option("header", "true")
         .option("inferSchema", "true")
         .load("/Volumes/dev_project/default/bronze/DataCoSupplyChainDataset.csv")
)

# ---------------------------------------
# Step 2 : Preview Data
# ---------------------------------------

df.show(5)

# ---------------------------------------
# Step 3 : Check Schema
# ---------------------------------------

df.printSchema()

# ---------------------------------------
# Step 4 : Validate Row Count
# ---------------------------------------

print(df.count())

# ---------------------------------------
# Step 5 : Validate Columns
# ---------------------------------------

print(len(df.columns))
print(df.columns)

# ---------------------------------------
# Step 6 : Check NULL Values
# ---------------------------------------

from pyspark.sql.functions import col, count, when

null_df = df.select([
    count(when(col(column).isNull(), column)).alias(column)
    for column in df.columns
])

null_df.show(truncate=False)

# ---------------------------------------
# Step 7 : Check Duplicate Records
# ---------------------------------------

total_rows = df.count()
distinct_rows = df.distinct().count()

print(f"Total Rows    : {total_rows}")
print(f"Distinct Rows : {distinct_rows}")
print(f"Duplicate Rows: {total_rows - distinct_rows}")

# ---------------------------------------
# Step 8 : Check Unique Business Keys
# ---------------------------------------

from pyspark.sql.functions import countDistinct

df.select(
    countDistinct("Order Id").alias("Unique_Orders"),
    countDistinct("Customer Id").alias("Unique_Customers"),
    countDistinct("Product Card Id").alias("Unique_Products")
).show()

# ---------------------------------------
# Step 9 : Check Data Types
# ---------------------------------------

df.printSchema()

# Order Status values
df.select("Order Status").distinct().show(truncate=False)

# Delivery Status values
df.select("Delivery Status").distinct().show(truncate=False)

# Shipping Mode values
df.select("Shipping Mode").distinct().show(truncate=False)

# ---------------------------------------
# Step 13 : Check Date Range
# ---------------------------------------

from pyspark.sql.functions import min, max

df.select(
    min("order date (DateOrders)").alias("First_Order_Date"),
    max("order date (DateOrders)").alias("Last_Order_Date"),
    min("shipping date (DateOrders)").alias("First_Shipping_Date"),
    max("shipping date (DateOrders)").alias("Last_Shipping_Date")
).show(truncate=False)

# ---------------------------------------
# Step 14 : Standardize Column Names
# ---------------------------------------

import re

def clean_column_name(column_name):
    column_name = column_name.lower()
    column_name = re.sub(r'[^a-z0-9]+', '_', column_name)
    column_name = re.sub(r'_+', '_', column_name)
    column_name = column_name.strip('_')
    return column_name

df_clean = df.toDF(*[clean_column_name(col) for col in df.columns])

df_clean.printSchema()

# ---------------------------------------
# Step 15 : Write Data to Bronze Delta Table
# ---------------------------------------

(
    # Use the cleaned DataFrame (column names are standardized)
    df_clean.write

        # Store the data in Delta Lake format
        # Delta provides ACID transactions, schema enforcement,
        # time travel, and better performance than CSV.
        .format("delta")

        # If the table already exists, replace it with the new data.
        # Useful during development because we run the notebook multiple times.
        .mode("overwrite")

        # Create a managed Delta table inside:
        # Catalog : dev_project
        # Schema  : default
        # Table   : bronze_supply_chain
        .saveAsTable("dev_project.default.bronze_supply_chain")
)

# ---------------------------------------
# Step 16 : Verify Bronze Table
# ---------------------------------------

# Count the number of records in the Bronze table.
# This ensures that all rows from the source CSV
# were successfully loaded into Delta.

spark.sql("""
SELECT COUNT(*) AS total_records
FROM dev_project.default.bronze_supply_chain
""").show()

# ---------------------------------------
# Step 17 : Preview Bronze Table
# ---------------------------------------

# Display the first 10 records from the Bronze table.
# This helps verify that the data looks correct.

spark.sql("""
SELECT *
FROM dev_project.default.bronze_supply_chain
LIMIT 10
""").show()

# ---------------------------------------
# Step 18 : Check Bronze Table Schema
# ---------------------------------------

# Display the schema of the Bronze Delta table.

spark.sql("""
DESCRIBE dev_project.default.bronze_supply_chain
""").show(truncate=False)